Clinical Trials Analysis

In [0]:
%sql
-- ============================================
-- 1. Clinical Trials Analysis 
-- ============================================

-- Problem 1: Trial Status Distribution
SELECT 
    trial_status_standardized,
    COUNT(*) as trial_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage,
    SUM(patient_count) as total_patients,  -- Changed from enrolled_patients
    ROUND(AVG(efficacy_rate), 2) as avg_efficacy
FROM pharma_catalog_prod.silver.silver_clinical_trials
GROUP BY trial_status_standardized
ORDER BY trial_count DESC;

-- Problem 2: Trial Phase Distribution
SELECT 
    trial_phase_standardized,
    COUNT(*) as trial_count,
    ROUND(AVG(patient_count), 0) as avg_patients,  -- Changed from enrolled_patients
    ROUND(AVG(efficacy_rate), 2) as avg_efficacy,
    ROUND(AVG(adverse_events_count), 2) as avg_adverse_events
FROM pharma_catalog_prod.silver.silver_clinical_trials
GROUP BY trial_phase_standardized
ORDER BY trial_phase_standardized;

-- Problem 3: Top 10 Drugs by Efficacy
SELECT 
    drug_name,
    condition_treated,
    ROUND(AVG(efficacy_rate), 2) as avg_efficacy,
    COUNT(*) as trial_count,
    ROUND(AVG(adverse_events_count), 2) as avg_adverse_events
FROM pharma_catalog_prod.silver.silver_clinical_trials
GROUP BY drug_name, condition_treated
ORDER BY avg_efficacy DESC
LIMIT 10;

-- Problem 4: Clinical Trials by Sponsor
SELECT 
    sponsor,
    COUNT(*) as trial_count,
    SUM(patient_count) as total_patients,  -- Changed from enrolled_patients
    ROUND(AVG(efficacy_rate), 2) as avg_efficacy
FROM pharma_catalog_prod.silver.silver_clinical_trials
GROUP BY sponsor
ORDER BY trial_count DESC;

-- Problem 5: Adverse Events by Trial Phase
SELECT 
    trial_phase_standardized,
    ROUND(AVG(adverse_events_count), 2) as avg_adverse_events,
    ROUND(AVG(serious_adverse_events), 2) as avg_serious_events,
    ROUND(SUM(adverse_events_count), 2) as total_adverse_events,
    ROUND(AVG(patient_count), 0) as avg_patients  -- Changed from enrolled_patients
FROM pharma_catalog_prod.silver.silver_clinical_trials
GROUP BY trial_phase_standardized
ORDER BY avg_adverse_events DESC;

-- Problem 6: Trials with High Efficacy (>80%) and Low Adverse Events (<20)
SELECT 
    trial_id,
    drug_name,
    condition_treated,
    trial_phase_standardized,
    efficacy_rate,
    adverse_events_count,
    patient_count,  -- Changed from enrolled_patients
    sponsor
FROM pharma_catalog_prod.silver.silver_clinical_trials
WHERE efficacy_rate > 80 AND adverse_events_count < 20
ORDER BY efficacy_rate DESC;

-- Problem 7: Budget vs Efficacy Analysis
SELECT 
    drug_name,
    ROUND(AVG(budget_millions), 2) as avg_budget,
    ROUND(AVG(efficacy_rate), 2) as avg_efficacy,
    ROUND(AVG(patient_count), 0) as avg_patients,  -- Changed from enrolled_patients
    COUNT(*) as trial_count
FROM pharma_catalog_prod.silver.silver_clinical_trials
GROUP BY drug_name
HAVING COUNT(*) > 1
ORDER BY avg_efficacy DESC;

-- Problem 8: Efficacy Rate by Enrollment Category
SELECT 
    enrollment_category,
    COUNT(*) as trial_count,
    ROUND(AVG(efficacy_rate), 2) as avg_efficacy,
    ROUND(AVG(adverse_events_count), 2) as avg_adverse_events,
    ROUND(AVG(completion_rate), 2) as avg_completion_rate
FROM pharma_catalog_prod.silver.silver_clinical_trials
GROUP BY enrollment_category
ORDER BY avg_efficacy DESC;

-- Problem 9: Publication Status Distribution
SELECT 
    publication_status,
    COUNT(*) as trial_count,
    ROUND(AVG(efficacy_rate), 2) as avg_efficacy,
    ROUND(AVG(data_quality_score), 2) as avg_data_quality
FROM pharma_catalog_prod.silver.silver_clinical_trials
GROUP BY publication_status
ORDER BY trial_count DESC;

-- Problem 10: Top Sponsors by Success Rate (Completed Trials)
SELECT 
    sponsor,
    COUNT(*) as total_trials,
    SUM(CASE WHEN trial_status_standardized = 'Completed' THEN 1 ELSE 0 END) as completed_trials,
    ROUND(SUM(CASE WHEN trial_status_standardized = 'Completed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as success_rate,
    ROUND(AVG(efficacy_rate), 2) as avg_efficacy
FROM pharma_catalog_prod.silver.silver_clinical_trials
GROUP BY sponsor
ORDER BY success_rate DESC;


Sales & Revenue Analysis

In [0]:
%sql
-- ============================================
-- 2. Sales & Revenue Analysis
-- ============================================

-- Problem 6: Sales by Region
SELECT 
    region,
    COUNT(*) as sale_count,
    ROUND(SUM(total_revenue_usd), 2) as total_revenue,
    ROUND(AVG(total_revenue_usd), 2) as avg_sale_value,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_sales
GROUP BY region
ORDER BY total_revenue DESC;

-- Problem 7: Sales by Channel
SELECT 
    sales_channel_standardized,
    COUNT(*) as sale_count,
    ROUND(SUM(total_revenue_usd), 2) as total_revenue,
    ROUND(AVG(total_revenue_usd), 2) as avg_sale_value,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_sales
GROUP BY sales_channel_standardized
ORDER BY total_revenue DESC;

-- Problem 8: Top 10 Drugs by Revenue
SELECT 
    drug_name,
    COUNT(*) as sale_count,
    ROUND(SUM(total_revenue_usd), 2) as total_revenue,
    ROUND(AVG(unit_price_usd), 2) as avg_price,
    SUM(units_sold) as total_units
FROM pharma_catalog_prod.silver.silver_sales
GROUP BY drug_name
ORDER BY total_revenue DESC
LIMIT 10;

-- Problem 9: Monthly Revenue Trend
SELECT 
    YEAR(sales_date) as year,
    MONTH(sales_date) as month,
    COUNT(*) as sale_count,
    ROUND(SUM(total_revenue_usd), 2) as total_revenue,
    ROUND(AVG(total_revenue_usd), 2) as avg_sale
FROM pharma_catalog_prod.silver.silver_sales
GROUP BY YEAR(sales_date), MONTH(sales_date)
ORDER BY year DESC, month DESC
LIMIT 12;

-- Problem 10: High Value Sales Analysis
SELECT 
    drug_name,
    region,
    sales_channel_standardized,
    ROUND(total_revenue_usd, 2) as revenue,
    sales_date
FROM pharma_catalog_prod.silver.silver_sales
WHERE is_high_value = true
ORDER BY total_revenue_usd DESC
LIMIT 20;


Inventory Analysis

In [0]:
%sql
-- ============================================
-- 3. Inventory Analysis
-- ============================================

-- Problem 11: Stock Status Distribution
SELECT 
    stock_status,
    COUNT(*) as inventory_count,
    SUM(quantity_in_stock) as total_quantity,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_inventory
GROUP BY stock_status
ORDER BY inventory_count DESC;

-- Problem 12: Expiry Risk Analysis
SELECT 
    expiry_risk,
    COUNT(*) as batch_count,
    SUM(quantity_in_stock) as total_quantity,
    ROUND(AVG(days_to_expiry), 0) as avg_days_to_expiry,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_inventory
GROUP BY expiry_risk
ORDER BY 
    CASE expiry_risk
        WHEN 'Critical' THEN 1
        WHEN 'High' THEN 2
        WHEN 'Medium' THEN 3
        ELSE 4
    END;

-- Problem 13: Warehouse Inventory Summary
SELECT 
    warehouse_location,
    COUNT(*) as batch_count,
    SUM(quantity_in_stock) as total_quantity,
    ROUND(AVG(quantity_in_stock), 0) as avg_quantity,
    ROUND(AVG(batch_quality_score), 2) as avg_quality_score
FROM pharma_catalog_prod.silver.silver_inventory
GROUP BY warehouse_location
ORDER BY total_quantity DESC;

-- Problem 14: Critical Stock Items
SELECT 
    drug_name,
    warehouse_location,
    quantity_in_stock,
    reorder_level,
    safety_stock,
    stock_status,
    stockout_risk_standardized
FROM pharma_catalog_prod.silver.silver_inventory
WHERE stock_status = 'Critical Stock' OR stock_status = 'Below Reorder Level'
ORDER BY quantity_in_stock ASC;

-- Problem 15: Supplier Performance
SELECT 
    supplier,
    COUNT(*) as batch_count,
    ROUND(AVG(lead_time_days), 2) as avg_lead_time,
    ROUND(AVG(batch_quality_score), 2) as avg_quality_score,
    SUM(quantity_in_stock) as total_inventory
FROM pharma_catalog_prod.silver.silver_inventory
GROUP BY supplier
ORDER BY avg_quality_score DESC;


Prescription Analytics

In [0]:
%sql
-- ============================================
-- 4. Prescription Analytics
-- ============================================

-- Problem 16: Prescription Frequency Distribution
SELECT 
    frequency_standardized,
    COUNT(*) as prescription_count,
    ROUND(AVG(duration_days), 2) as avg_duration,
    ROUND(AVG(refill_count), 2) as avg_refills,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_prescriptions
GROUP BY frequency_standardized
ORDER BY prescription_count DESC;

-- Problem 17: Adherence Level Distribution
SELECT 
    adherence_level,
    COUNT(*) as patient_count,
    ROUND(AVG(adherence_score), 2) as avg_adherence_score,
    ROUND(AVG(duration_days), 2) as avg_duration,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_prescriptions
GROUP BY adherence_level
ORDER BY adherence_level DESC;

-- Problem 18: Most Prescribed Drugs
SELECT 
    drug_name,
    COUNT(*) as prescription_count,
    ROUND(AVG(dosage_mg), 2) as avg_dosage,
    ROUND(AVG(duration_days), 2) as avg_duration,
    ROUND(AVG(refill_count), 2) as avg_refills
FROM pharma_catalog_prod.silver.silver_prescriptions
GROUP BY drug_name
ORDER BY prescription_count DESC
LIMIT 10;

-- Problem 19: Prescription by Insurance Provider
SELECT 
    insurance_provider,
    COUNT(*) as prescription_count,
    ROUND(AVG(copay_amount), 2) as avg_copay,
    ROUND(AVG(adherence_score), 2) as avg_adherence
FROM pharma_catalog_prod.silver.silver_prescriptions
GROUP BY insurance_provider
ORDER BY prescription_count DESC;

-- Problem 20: Duration Category Distribution
SELECT 
    duration_category,
    COUNT(*) as prescription_count,
    ROUND(AVG(adherence_score), 2) as avg_adherence,
    ROUND(AVG(refill_count), 2) as avg_refills,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_prescriptions
GROUP BY duration_category
ORDER BY duration_category;


Side Effects Analysis

In [0]:
%sql
-- ============================================
-- 5. Side Effects Analysis
-- ============================================

-- Problem 21: Severity Distribution
SELECT 
    severity_standardized,
    COUNT(*) as event_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage,
    ROUND(AVG(patient_age), 2) as avg_patient_age,
    SUM(CASE WHEN is_fatal = true THEN 1 ELSE 0 END) as fatal_events
FROM pharma_catalog_prod.silver.silver_side_effects
GROUP BY severity_standardized
ORDER BY event_count DESC;

-- Problem 22: Most Common Side Effects
SELECT 
    reaction_meddra_term,
    COUNT(*) as event_count,
    COUNT(DISTINCT drug_name) as unique_drugs,
    ROUND(AVG(severity_score), 2) as avg_severity,
    SUM(CASE WHEN is_fatal = true THEN 1 ELSE 0 END) as fatal_events
FROM pharma_catalog_prod.silver.silver_side_effects
GROUP BY reaction_meddra_term
ORDER BY event_count DESC
LIMIT 10;

-- Problem 23: Side Effects by Drug
SELECT 
    drug_name,
    COUNT(*) as event_count,
    ROUND(AVG(severity_score), 2) as avg_severity,
    COUNT(DISTINCT reaction_meddra_term) as unique_reactions,
    SUM(CASE WHEN is_serious = true THEN 1 ELSE 0 END) as serious_events
FROM pharma_catalog_prod.silver.silver_side_effects
GROUP BY drug_name
ORDER BY event_count DESC
LIMIT 10;

-- Problem 24: Age Group Analysis
SELECT 
    age_group,
    COUNT(*) as event_count,
    ROUND(AVG(severity_score), 2) as avg_severity,
    SUM(CASE WHEN is_serious = true THEN 1 ELSE 0 END) as serious_events,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_side_effects
GROUP BY age_group
ORDER BY age_group;

-- Problem 25: Reporter Type Distribution
SELECT 
    reporter_type,
    COUNT(*) as event_count,
    ROUND(AVG(severity_score), 2) as avg_severity,
    SUM(CASE WHEN is_serious = true THEN 1 ELSE 0 END) as serious_events
FROM pharma_catalog_prod.silver.silver_side_effects
GROUP BY reporter_type
ORDER BY event_count DESC;


R&D Pipeline Analysis

In [0]:
%sql
-- ============================================
-- 6. R&D Pipeline Analysis
-- ============================================

-- Problem 26: Development Stage Distribution
SELECT 
    development_stage_standardized,
    COUNT(*) as project_count,
    ROUND(AVG(project_cost_millions), 2) as avg_cost,
    SUM(project_cost_millions) as total_cost,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_rd_pipeline
GROUP BY development_stage_standardized
ORDER BY development_stage_standardized;

-- Problem 27: Risk Assessment Distribution
SELECT 
    risk_assessment,
    COUNT(*) as project_count,
    ROUND(AVG(project_cost_millions), 2) as avg_cost,
    SUM(project_cost_millions) as total_cost,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_rd_pipeline
GROUP BY risk_assessment
ORDER BY project_count DESC;

-- Problem 28: Funding Source Analysis
SELECT 
    funding_source,
    COUNT(*) as project_count,
    ROUND(AVG(project_cost_millions), 2) as avg_cost,
    SUM(project_cost_millions) as total_cost,
    ROUND(AVG(team_size), 2) as avg_team_size
FROM pharma_catalog_prod.silver.silver_rd_pipeline
GROUP BY funding_source
ORDER BY total_cost DESC;

-- Problem 29: Disease Area Focus
SELECT 
    disease_area,
    COUNT(*) as project_count,
    ROUND(AVG(project_cost_millions), 2) as avg_cost,
    ROUND(AVG(team_size), 2) as avg_team_size,
    SUM(CASE WHEN patent_filed = 'Yes' THEN 1 ELSE 0 END) as patent_count
FROM pharma_catalog_prod.silver.silver_rd_pipeline
GROUP BY disease_area
ORDER BY project_count DESC;

-- Problem 30: Priority Level Distribution
SELECT 
    priority_level_standardized,
    COUNT(*) as project_count,
    ROUND(AVG(project_cost_millions), 2) as avg_cost,
    SUM(project_cost_millions) as total_cost,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_rd_pipeline
GROUP BY priority_level_standardized
ORDER BY priority_level_standardized;


Physician Prescribing Analysis

In [0]:
%sql
-- ============================================
-- 7. Physician Prescribing Analysis
-- ============================================

-- Problem 31: Experience Level Distribution
SELECT 
    experience_level,
    COUNT(*) as physician_count,
    ROUND(AVG(prescription_volume_monthly), 2) as avg_prescriptions,
    ROUND(AVG(patient_satisfaction_score), 2) as avg_satisfaction,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_physician
GROUP BY experience_level
ORDER BY experience_level;

-- Problem 32: Specialty Distribution
SELECT 
    specialty,
    COUNT(*) as physician_count,
    ROUND(AVG(prescription_volume_monthly), 2) as avg_prescriptions,
    ROUND(AVG(patient_satisfaction_score), 2) as avg_satisfaction,
    SUM(CASE WHEN e_prescribing_flag = true THEN 1 ELSE 0 END) as e_prescribing_count
FROM pharma_catalog_prod.silver.silver_physician
GROUP BY specialty
ORDER BY physician_count DESC;

-- Problem 33: Prescription Volume by Practice Type
SELECT 
    practice_type,
    COUNT(*) as physician_count,
    ROUND(AVG(prescription_volume_monthly), 2) as avg_prescriptions,
    ROUND(AVG(patient_satisfaction_score), 2) as avg_satisfaction,
    ROUND(AVG(off_label_prescribing_rate), 2) as avg_off_label_rate
FROM pharma_catalog_prod.silver.silver_physician
GROUP BY practice_type
ORDER BY avg_prescriptions DESC;

-- Problem 34: Top Prescribing Physicians
SELECT 
    physician_name,
    specialty,
    practice_type,
    prescription_volume_monthly,
    patient_satisfaction_score,
    off_label_prescribing_rate
FROM pharma_catalog_prod.silver.silver_physician
ORDER BY prescription_volume_monthly DESC
LIMIT 20;

-- Problem 35: Satisfaction vs Prescription Volume
SELECT 
    satisfaction_category,
    COUNT(*) as physician_count,
    ROUND(AVG(prescription_volume_monthly), 2) as avg_prescriptions,
    ROUND(AVG(off_label_prescribing_rate), 2) as avg_off_label_rate
FROM pharma_catalog_prod.silver.silver_physician
GROUP BY satisfaction_category
ORDER BY satisfaction_category;


Marketing Analysis

In [0]:
%sql
-- ============================================
-- 8. Marketing Analysis
-- ============================================

-- Problem 36: ROI Category Distribution
SELECT 
    roi_category,
    COUNT(*) as campaign_count,
    ROUND(AVG(budget_usd), 2) as avg_budget,
    ROUND(AVG(roi_percent), 2) as avg_roi,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM pharma_catalog_prod.silver.silver_marketing
GROUP BY roi_category
ORDER BY avg_roi DESC;

-- Problem 37: Channel Performance
SELECT 
    channel,
    COUNT(*) as campaign_count,
    ROUND(AVG(budget_usd), 2) as avg_budget,
    ROUND(AVG(roi_percent), 2) as avg_roi,
    ROUND(AVG(conversion_rate), 2) as avg_conversion_rate
FROM pharma_catalog_prod.silver.silver_marketing
GROUP BY channel
ORDER BY avg_roi DESC;

-- Problem 38: Target Audience Analysis
SELECT 
    target_audience,
    COUNT(*) as campaign_count,
    ROUND(AVG(budget_usd), 2) as avg_budget,
    ROUND(AVG(roi_percent), 2) as avg_roi,
    ROUND(AVG(engagement_rate), 2) as avg_engagement
FROM pharma_catalog_prod.silver.silver_marketing
GROUP BY target_audience
ORDER BY avg_roi DESC;

-- Problem 39: Seasonal Campaign Performance
SELECT 
    seasonal,
    COUNT(*) as campaign_count,
    ROUND(AVG(budget_usd), 2) as avg_budget,
    ROUND(AVG(roi_percent), 2) as avg_roi,
    ROUND(AVG(conversion_rate), 2) as avg_conversion_rate
FROM pharma_catalog_prod.silver.silver_marketing
GROUP BY seasonal
ORDER BY avg_roi DESC;

-- Problem 40: Drug Type Campaign Performance
SELECT 
    drug_type,
    COUNT(*) as campaign_count,
    ROUND(AVG(budget_usd), 2) as avg_budget,
    ROUND(AVG(roi_percent), 2) as avg_roi,
    ROUND(AVG(conversion_rate), 2) as avg_conversion_rate
FROM pharma_catalog_prod.silver.silver_marketing
GROUP BY drug_type
ORDER BY avg_roi DESC;
